# OpenClaw Colab LLM Launcher

Run these cells from top to bottom. The notebook pulls the latest launcher scripts from GitHub, builds native `llama-server`, starts the model with `--jinja`, exposes it through Cloudflare, and checks whether the endpoint returns real OpenAI-style `tool_calls`.

In [1]:
# 1. Clone or update the launcher repo
REPO_URL = "https://github.com/Cobster-10/custom_llm.git"
REPO_DIR = "/content/custom_llm"

import os
from pathlib import Path

if Path(REPO_DIR, ".git").exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

Cloning into '/content/custom_llm'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 6.22 KiB | 6.22 MiB/s, done.
/content/custom_llm


In [2]:
# 2. Choose model/runtime settings
# Q8_K_P is the 8-bit quantization and fits large-memory Colab GPUs.
# If Colab gives you less VRAM, try Q4_K_M, IQ3_M, or IQ2_M.
%env MODEL_REPO=HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
%env MODEL_QUANT=Q8_K_P
%env PORT=8080
%env CTX_SIZE=131072
%env N_GPU_LAYERS=99
%env PARALLEL=1

env: MODEL_REPO=HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
env: MODEL_QUANT=Q8_K_P
env: PORT=8080
env: CTX_SIZE=131072
env: N_GPU_LAYERS=99
env: PARALLEL=1


In [4]:
# 3. Install dependencies, build llama.cpp with CUDA, and install cloudflared
!bash scripts/setup_colab.sh

bash: scripts/setup_colab.sh: No such file or directory


In [ ]:
# 4. Start llama-server in the background
!bash scripts/start_llama_server.sh

In [ ]:
# 5. Local smoke test: does the server return structured OpenAI tool_calls?
!python tests/test_tool_call.py --base-url http://127.0.0.1:8080/v1 --model "$MODEL_REPO:$MODEL_QUANT"

In [ ]:
# 6. Start Cloudflare tunnel and print the public OpenAI-compatible base URL
!bash scripts/start_cloudflare_tunnel.sh

In [ ]:
# 7. Public smoke test. Use this same base URL in OpenClaw if it passes.
from pathlib import Path

base_url_path = Path('/content/openclaw_base_url.txt')
if not base_url_path.exists():
    raise RuntimeError('No Cloudflare URL found. Re-run the tunnel cell first.')

public_base_url = base_url_path.read_text().strip()
print('OpenClaw base URL:', public_base_url)
!python tests/test_tool_call.py --base-url {public_base_url} --model "$MODEL_REPO:$MODEL_QUANT"

In [ ]:
# Optional: stop background services
!test -f /content/cloudflared.pid && kill $(cat /content/cloudflared.pid) || true
!test -f /content/llama-server.pid && kill $(cat /content/llama-server.pid) || true